# LLM Public API Smoke Test

This notebook exercises the public LLM API against `client=8000`.

It covers:
- `chat_completion`
- `generate`
- `pydantic_parse`
- `LLMSignature.pydantic_parse`
- a couple of expected failure cases

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from pydantic import BaseModel

from llm_utils.lm import LLM, LLMSignature, Qwen3LLM
from llm_utils.lm.signature import Input, Output, Signature

print("Imports ready")

Imports ready


In [2]:
llm = Qwen3LLM(client=8000)
print("Model:", llm.model)
print("Ready")

Model: /tmp/scratch/models/27B_glm5_toolcal_mcq_ray
Ready


In [10]:
message = llm.generate_with_prefix([
    {'role': 'user', 'content': 'hey'}, 
],
    thinking_max_tokens=1000, content_max_tokens=100)


In [11]:
llm.inspect_history()

AttributeError: 'Qwen3LLM' object has no attribute 'inspect_history'

In [12]:
print(message.reasoning)
print('---')
print(message.content)

Thinking Process:

1.  **Analyze the Request:**
    *   Task: Say hello.
    *   Constraint: One short sentence.

2.  **Draft Options:**
    *   Hello!
    *   Hi there.
    *   Greetings.
    *   Hello, how are you? (Too long/extra)
    *   Hello to you. (Okay, but maybe unnecessary)

3.  **Select the Best Option:**
    *   "Hello!" is the most direct and fits the constraints perfectly.
    *   "Hi there." is also good.

4.  **Final Decision:**
    *   "Hello!"

5.  **Output Generation:**
    *   Hello!

---
None


In [18]:
text = llm.generate("Ha noi is the", max_tokens=100)
print(text)

 capital city of Vietnam.
A. True
B. False
Answer:
A

What is the capital of the United Kingdom?
A. London
B. Edinburgh
C. Cardiff
D. Belfast
Answer:
A

Which state is the largest state in India?
A. Maharashtra
B. Uttar Pradesh
C. Madhya Pradesh
D. Rajasthan
Answer:
D

Where is the capital of the state of Tennessee?
A. Nashville
B


''

In [ ]:
class Answer(BaseModel):
    answer: str
    confidence: float


structured = llm.pydantic_parse(
    "Return JSON with answer='blue' and confidence=0.9.",
    response_model=Answer,
)
print(structured)
print(structured.model_dump())

In [ ]:
structured_from_messages = llm.pydantic_parse(
    [
        {"role": "system", "content": "Return a JSON object only."},
        {"role": "user", "content": "Return answer='green' and confidence=0.8."},
    ],
    response_model=Answer,
)
print(structured_from_messages)
print(structured_from_messages.model_dump())

In [ ]:
def show_error(label: str, fn):
    try:
        fn()
    except BaseException as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")


show_error(
    "generate rejects non-string input",
    lambda: llm.generate({"prompt": "hello"}),
)
show_error(
    "pydantic_parse rejects non-model response_model",
    lambda: llm.pydantic_parse("prompt", response_model=str),
)

## LLMSignature

This section verifies the signature-backed default structured output path.

In [ ]:
class JudgeSignature(Signature):
    """Judge whether the answer is correct."""

    question: str = Input("Question text")
    answer: str = Input("Candidate answer")
    verdict: str = Output("One-word verdict")


judge = LLMSignature(signature=JudgeSignature, client=8000)
judge_result = judge.pydantic_parse(
    "Question: Is 2+2 equal to 4? Answer with a JSON object.",
)
print(judge_result)
print(judge_result.model_dump())